### Load csv into pandas dataframe, parse date and time into single timestamp, filter out unimportant columns and low confidence records.

In [209]:
import pandas as pd
from pathlib import Path

date = "2026-05-02"

file_path = Path.cwd().parent.joinpath(
    "data/raw", f"VIIRS_SNNP_NRT_world_14days_{date}.csv"
)

data = pd.read_csv(file_path)
# data["acq_date"] = pd.to_datetime(data["acq_date"], format="%Y-%m-%d")
# TODO remove
# transform time in a string with actual utc mil time format
# data["acq_time"] = data["acq_time"].apply(lambda x: f"{x:04d}")
# data["timestamp"] = pd.to_datetime(
#     data["acq_date"] + data["acq_time"], format="%Y-%m-%d%H%M", utc=True
# )
# data.drop(
#     columns=["acq_date", "satellite", "instrument", "version", "acq_time"], inplace=True
# )

data.drop(columns=["satellite", "instrument", "version", "acq_time"], inplace=True)

data = data[data["confidence"] != "l"]

data.info()
data.head()


<class 'pandas.DataFrame'>
Index: 329115 entries, 0 to 379757
Data columns (total 10 columns):
 #   Column      Non-Null Count   Dtype  
---  ------      --------------   -----  
 0   latitude    329115 non-null  float64
 1   longitude   329115 non-null  float64
 2   bright_ti4  329115 non-null  float64
 3   scan        329115 non-null  float64
 4   track       329115 non-null  float64
 5   acq_date    329115 non-null  str    
 6   confidence  329115 non-null  str    
 7   bright_ti5  329115 non-null  float64
 8   frp         329115 non-null  float64
 9   daynight    329115 non-null  str    
dtypes: float64(7), str(3)
memory usage: 27.6 MB


,latitude,longitude,bright_ti4,scan,track,acq_date,confidence,bright_ti5,frp,daynight
0,32.33215,44.09279,306.72,0.71,0.75,2026-05-01,n,288.92,2.80,N
1,32.88856,35.09304,296.63,0.44,0.46,2026-05-01,n,281.80,1.02,N
2,33.15357,44.78751,302.25,0.73,0.76,2026-05-01,n,287.02,2.17,N
3,33.15594,44.77987,316.28,0.73,0.76,2026-05-01,n,288.05,2.17,N
4,33.15751,44.78342,342.93,0.73,0.76,2026-05-01,n,289.27,6.42,N


### Load other data into geodataframe

In [210]:
import geopandas as gpd

fire_data = gpd.GeoDataFrame(
    data, geometry=gpd.points_from_xy(data.longitude, data.latitude), crs="EPSG:4326"
)

countries_area = pd.read_csv(Path.cwd().parent.joinpath("data/raw", "surface_area.csv"))

countries_boundaries = gpd.read_file(
    Path.cwd().parent.joinpath("data/raw", "geoboundaries_world.geojson")
).to_crs("EPSG:4326")

# check for invalid geometries and repair them if necessary
if not countries_boundaries.is_valid.all():
    countries_boundaries.make_valid()
    print("Repairing invalid geometries")
if not countries_boundaries.is_valid.all():
    raise Exception("Some geometries are invalid and not reparable")
countries_boundaries.info()

<class 'geopandas.geodataframe.GeoDataFrame'>
RangeIndex: 218 entries, 0 to 217
Data columns (total 4 columns):
 #   Column      Non-Null Count  Dtype   
---  ------      --------------  -----   
 0   shapeGroup  218 non-null    str     
 1   shapeType   218 non-null    str     
 2   shapeName   218 non-null    str     
 3   geometry    218 non-null    geometry
dtypes: geometry(1), str(3)
memory usage: 6.9 KB


### Filter out attributes in country area and polygons, cast strings into correct types
Also keep only area data from the last surveyed year (2023).

In [211]:
countries_area = countries_area[countries_area["TIME_PERIOD"] == 2023]
countries_area = countries_area[["REF_AREA", "OBS_VALUE"]]
countries_area["OBS_VALUE"] = countries_area["OBS_VALUE"].astype(float)
countries_area = countries_area.rename(
    columns={"OBS_VALUE": "country_area", "REF_AREA": "iso_code"}
)
countries_boundaries = countries_boundaries.rename(
    columns={"shapeGroup": "iso_code", "shapeName": "name"}
)
countries_boundaries = countries_boundaries.drop(columns=["shapeType"])
countries_area.dropna()
countries_area.info()

<class 'pandas.DataFrame'>
Index: 215 entries, 1744 to 9791
Data columns (total 2 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   iso_code      215 non-null    str    
 1   country_area  215 non-null    float64
dtypes: float64(1), str(1)
memory usage: 5.0 KB


### Compute true pixel area for fires pixels

In [212]:
fire_data["fire_area"] = fire_data["scan"] * fire_data["track"]
fire_data = fire_data[["acq_date", "geometry", "fire_area"]]
fire_data.info()

<class 'geopandas.geodataframe.GeoDataFrame'>
Index: 329115 entries, 0 to 379757
Data columns (total 3 columns):
 #   Column     Non-Null Count   Dtype   
---  ------     --------------   -----   
 0   acq_date   329115 non-null  str     
 1   geometry   329115 non-null  geometry
 2   fire_area  329115 non-null  float64 
dtypes: float64(1), geometry(1), str(1)
memory usage: 10.0 MB


### Join countries polygons with area table

In [213]:
countries_boundaries_area = pd.merge(
    countries_boundaries,
    countries_area,
    left_on="iso_code",
    right_on="iso_code",
    how="left",
)
# countries_boundaries_area = countries_boundaries_area.drop(columns="iso_code")
countries_boundaries_area.info()


selected = countries_boundaries_area.loc[
    countries_boundaries_area["country_area"].isna()
]
selected

<class 'geopandas.geodataframe.GeoDataFrame'>
RangeIndex: 218 entries, 0 to 217
Data columns (total 4 columns):
 #   Column        Non-Null Count  Dtype   
---  ------        --------------  -----   
 0   iso_code      218 non-null    str     
 1   name          218 non-null    str     
 2   geometry      218 non-null    geometry
 3   country_area  194 non-null    float64 
dtypes: float64(1), geometry(1), str(2)
memory usage: 6.9 KB


,iso_code,name,geometry,country_area
5,ATA,Antarctica,"MULTIPOLYGON (((-60.06171 -79.6813, -60.05473 ...",NaN
94,XKX,Kosovo,"POLYGON ((20.59429 41.87733, 20.5955 41.8765, ...",NaN
168,TWN,Taiwan,"MULTIPOLYGON (((116.71997 20.70818, 116.71902 ...",NaN
179,VAT,Vatican City,"POLYGON ((12.4538 41.90682, 12.45308 41.90668,...",NaN
198,111,Abyei,"POLYGON ((29 9.67356, 29 10.16667, 27.83333 10...",NaN
199,112,Aksai Chin,"MULTIPOLYGON (((78.69839 34.09307, 78.69837 34...",NaN
200,113,CH-IN,"MULTIPOLYGON (((79.70073 30.97073, 79.70088 30...",NaN
201,114,Demchok,"POLYGON ((79.15197 33.18187, 79.15271 33.18106...",NaN
202,115,Dragonja,"MULTIPOLYGON (((13.67648 45.44426, 13.67648 45...",NaN
203,116,Dramana-Shakatoe,"POLYGON ((89.12804 27.61484, 89.12799 27.61484...",NaN


As we can observe the countries with a geometry coming from the the countries boundaries but without an area are those not officially recognized by UN. Since we need to later compute the percentage of the land surface affected by wildfires, we will drop them

In [214]:
countries_boundaries_area = countries_boundaries_area.dropna()
countries_boundaries_area.info()

<class 'geopandas.geodataframe.GeoDataFrame'>
Index: 194 entries, 0 to 197
Data columns (total 4 columns):
 #   Column        Non-Null Count  Dtype   
---  ------        --------------  -----   
 0   iso_code      194 non-null    str     
 1   name          194 non-null    str     
 2   geometry      194 non-null    geometry
 3   country_area  194 non-null    float64 
dtypes: float64(1), geometry(1), str(2)
memory usage: 7.6 KB


### Spatially join countries with fires. Merge and group fire data by date and countries.

In [ ]:
fire_data_countries = gpd.sjoin(
    countries_boundaries_area,
    fire_data,
    how="left",
)

# dissolve() times out and crashes the kernel, worked around by using df.groupby() and joining
# back later with the countries geometries on the iso code key
fire_data_by_countries_dates = pd.DataFrame(
    fire_data_countries.groupby(by=["iso_code", "acq_date", "name", "country_area"])[
        "fire_area"
    ]
    .sum()
    .reset_index()
)
fire_data_by_countries = pd.DataFrame(
    fire_data_countries[["iso_code", "name", "country_area", "fire_area"]]
    .groupby(by=["iso_code", "name", "country_area"])["fire_area"]
    .sum()
    .reset_index()
)

fire_data_by_countries = gpd.GeoDataFrame(
    fire_data_by_countries.merge(
        countries_boundaries, on=["iso_code", "name"], how="left"
    )
)

fire_data_by_countries.info()

<class 'geopandas.geodataframe.GeoDataFrame'>
RangeIndex: 194 entries, 0 to 193
Data columns (total 5 columns):
 #   Column        Non-Null Count  Dtype   
---  ------        --------------  -----   
 0   iso_code      194 non-null    str     
 1   name          194 non-null    str     
 2   country_area  194 non-null    float64 
 3   fire_area     194 non-null    float64 
 4   geometry      194 non-null    geometry
dtypes: float64(2), geometry(1), str(2)
memory usage: 7.7 KB


### Compute percentage of wild fires area

In [216]:
fire_data_by_countries["area_perc"] = (
    fire_data_by_countries["fire_area"] / fire_data_by_countries["country_area"]
) * 100
fire_data_by_countries_dates["area_perc"] = (
    fire_data_by_countries_dates["fire_area"]
    / fire_data_by_countries_dates["country_area"]
) * 100
fire_data_by_countries.info()
fire_data_by_countries_dates

<class 'geopandas.geodataframe.GeoDataFrame'>
RangeIndex: 194 entries, 0 to 193
Data columns (total 6 columns):
 #   Column        Non-Null Count  Dtype   
---  ------        --------------  -----   
 0   iso_code      194 non-null    str     
 1   name          194 non-null    str     
 2   country_area  194 non-null    float64 
 3   fire_area     194 non-null    float64 
 4   geometry      194 non-null    geometry
 5   area_perc     194 non-null    float64 
dtypes: float64(3), geometry(1), str(2)
memory usage: 9.2 KB


,iso_code,acq_date,name_x,country_area,fire_area,name_y,geometry,area_perc
0,AFG,2026-04-19,Afghanistan,652870.0,0.3180,Afghanistan,"POLYGON ((74.88986 37.23409, 74.88957 37.23435...",0.000049
1,AFG,2026-04-20,Afghanistan,652870.0,0.4066,Afghanistan,"POLYGON ((74.88986 37.23409, 74.88957 37.23435...",0.000062
2,AFG,2026-04-24,Afghanistan,652870.0,0.6912,Afghanistan,"POLYGON ((74.88986 37.23409, 74.88957 37.23435...",0.000106
3,AFG,2026-04-25,Afghanistan,652870.0,0.8336,Afghanistan,"POLYGON ((74.88986 37.23409, 74.88957 37.23435...",0.000128
4,AFG,2026-04-26,Afghanistan,652870.0,0.1517,Afghanistan,"POLYGON ((74.88986 37.23409, 74.88957 37.23435...",0.000023
...,...,...,...,...,...,...,...,...
1624,ZWE,2026-04-26,Zimbabwe,390760.0,15.1945,Zimbabwe,"POLYGON ((29.3753 -22.19547, 29.37594 -22.1953...",0.003888
1625,ZWE,2026-04-27,Zimbabwe,390760.0,10.1704,Zimbabwe,"POLYGON ((29.3753 -22.19547, 29.37594 -22.1953...",0.002603
1626,ZWE,2026-04-30,Zimbabwe,390760.0,5.6237,Zimbabwe,"POLYGON ((29.3753 -22.19547, 29.37594 -22.1953...",0.001439
1627,ZWE,2026-05-01,Zimbabwe,390760.0,15.2688,Zimbabwe,"POLYGON ((29.3753 -22.19547, 29.37594 -22.1953...",0.003907


### Transpose dates values to columns that contains the percentage of wildfire area for each day to reduce geometries

In [217]:
fire_data_by_countries_dates = fire_data_by_countries_dates.pivot_table(
    index="iso_code", columns="acq_date", values="area_perc"
)
fire_data_by_countries_dates = fire_data_by_countries_dates.fillna(0)

In [220]:
fire_data_by_countries_dates = gpd.GeoDataFrame(
    fire_data_by_countries_dates.merge(
        countries_boundaries, on=["iso_code", "iso_code"], how="left"
    )
)

In [221]:
fire_data_by_countries_dates

,iso_code,2026-04-19,2026-04-20,2026-04-21,2026-04-22,2026-04-23,2026-04-24,2026-04-25,2026-04-26,2026-04-27,2026-04-28,2026-04-30,2026-05-01,2026-05-02,name,geometry
0,AFG,0.000049,0.000062,0.000000,0.000000,0.000000,0.000106,0.000128,0.000023,0.000000,0.000000,0.000064,0.000000,0.000000,Afghanistan,"POLYGON ((74.88986 37.23409, 74.88957 37.23435..."
1,AGO,0.000945,0.001412,0.001267,0.001188,0.000354,0.000206,0.000483,0.000664,0.001254,0.000000,0.000000,0.000612,0.001142,Angola,"MULTIPOLYGON (((11.7163 -16.50801, 11.7151 -16..."
2,ALB,0.002239,0.000568,0.000000,0.000000,0.000000,0.000568,0.000000,0.001493,0.001408,0.000568,0.000000,0.001950,0.002784,Albania,"POLYGON ((20.0789 42.5558, 20.07785 42.55594, ..."
3,ARE,0.020586,0.014744,0.018782,0.062088,0.089780,0.017672,0.005476,0.011856,0.022111,0.000000,0.012234,0.011682,0.001785,United Arab Emirates,"MULTIPOLYGON (((53.12127 24.12094, 53.12033 24..."
4,ARG,0.000596,0.000062,0.000349,0.000786,0.000866,0.000682,0.000548,0.000761,0.001509,0.000000,0.001907,0.000508,0.000266,Argentina,"MULTIPOLYGON (((-63.83417 -54.68583, -63.84175..."
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
163,VUT,0.007039,0.003652,0.001152,0.022486,0.000000,0.000000,0.010897,0.021739,0.048254,0.000000,0.038258,0.306593,0.027719,Vanuatu,"MULTIPOLYGON (((169.76503 -20.13292, 169.76247..."
164,YEM,0.000139,0.000793,0.000854,0.000591,0.000236,0.000330,0.000387,0.000429,0.001324,0.000000,0.000287,0.000218,0.000000,Yemen,"MULTIPOLYGON (((53.28038 12.12789, 53.28024 12..."
165,ZAF,0.001382,0.001621,0.001821,0.000694,0.001660,0.001081,0.001626,0.003494,0.001263,0.000000,0.000362,0.001691,0.002007,South Africa,"MULTIPOLYGON (((37.72466 -46.82744, 37.71847 -..."
166,ZMB,0.000659,0.000585,0.001415,0.000535,0.000849,0.000786,0.000943,0.000699,0.001248,0.000000,0.000219,0.003533,0.001820,Zambia,"POLYGON ((23.43423 -17.63787, 23.43423 -17.637..."


### Simplify polygons
Using the non-simplified polygons created the resulting html map was huge and taking a performance hit. Unlike `simplify()`, `simplify_coverage()` assumes that the GeoSeries forms a polygonal coverage. Polygons borders remain thus consistent.

In [222]:
# TODO optimize memory: write to disk as soon as simplify is done and use del keyword to manually decrease reference count and make garbage collector deallocate the objects

fire_data_by_countries_simplified = fire_data_by_countries.copy()
fire_data_by_countries_simplified["geometry"] = (
    fire_data_by_countries.geometry.simplify_coverage(tolerance=0.05)
)
countries_boundaries_simplified = countries_boundaries.copy()
countries_boundaries_simplified["geometry"] = (
    countries_boundaries.geometry.simplify_coverage(tolerance=0.05)
)
fire_data_by_countries_dates_simplified = fire_data_by_countries_dates.copy()
fire_data_by_countries_dates_simplified["geometry"] = (
    fire_data_by_countries_dates_simplified.geometry.simplify_coverage(tolerance=0.05)
)

### Save processed data

In [223]:
fire_data_by_countries_simplified.to_file(
    Path.cwd().parent.joinpath("data/processed", "fire_data_by_countries.gpkg")
)
fire_data_by_countries_dates_simplified.to_file(
    Path.cwd().parent.joinpath("data/processed", "fire_data_by_countries_dates.gpkg")
)
countries_boundaries_simplified.to_file(
    Path.cwd().parent.joinpath("data/processed", "countries_boundaries_simplified.gpkg")
)